# **Hybrid RAG**
Combines semantic vector search with keyword-based search to retrieve information.
This allows RAG to find both conceptually similar content and exact keyword matches.

### **Components & Architecture**
- **Embedding Model:** `OpenAIEmbeddings`
- **Vector Database:** Chroma (Local) / Qdrant (Cloud)
- **Keyword Search:** BM25 (`BM25Retriever` - Sparse Search)
- **Retriever:** `EnsembleRetriever` (Hybrid Search)
- **LLM Model:** `ChatOpenAI`

## **Initial Setup**

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')

## **Indexing**

In [ ]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# load data
from langchain.document_loaders import CSVLoader
loader = CSVLoader("./context.csv")
documents = loader.load()

In [ ]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

## **Chroma Vector Database**

In [ ]:
# create vectorstore
from langchain.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents, embeddings)

## **Qdrant Vector Database (Optional)**

In [ ]:
# # optional vectorstore (Qdrant Cloud)
# from langchain_community.vectorstores import Qdrant
# vectorstore = Qdrant.from_documents(
#     documents,
#     embeddings,
#     url="your_qdrant_url",
#     prefer_grpc=True,
#     collection_name="hybrid_rag_collection",
#     api_key=os.environ.get("QDRANT_API_KEY"),
# )

## **Retrievers**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

### **Keyword Retriever**

In [ ]:
# create keyword retriever
from langchain.retrievers import BM25Retriever
keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k =  3

In [ ]:
# test keyword retriever
keyword_retriever.get_relevant_documents("what bacteria grow on macconkey agar")

### **Ensemble Retriever**

In [ ]:
# create ensemble retriever
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(retrievers=[retriever, keyword_retriever], weights=[0.5, 0.5])

In [ ]:
# test ensemble retriever
ensemble_retriever.get_relevant_documents("what bacteria grow on macconkey agar")

## **RAG Chain**

In [ ]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# create document chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

template = """"
You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.
Context: {context}

Question: {input}

Answer:

"""
prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": ensemble_retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# response
response = rag_chain.invoke('what bacteria grow on macconkey agar')
response